# Manual Upload Preprocessing

Normalize heterogeneous CSV files in `local_data/manual_upload/` into the pipeline extraction schema, then create a standardized manual dataset in `local_data/manual/`.

In [9]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path("src").resolve()))

from etl_pipeline.common.io import read_csv, write_csv, write_json
from etl_pipeline.extraction.manual_upload import run_manual_upload_extraction
from etl_pipeline.preprocessing.schema import STANDARDIZED_FIELDS
from etl_pipeline.preprocessing.transform import transform_extraction_rows

CONFIG = {
    "paths": {
        "local_data_dir": "local_data",
        "master_dir": "local_data/master",
        "logs_dir": "logs",
    },
    "manual_upload": {
        "input_dir": "local_data/manual_upload",
        "output_file": "local_data/manual/manual_extraction_raw.csv",
    },
}
RUN_ID = "manual_upload_notebook"
MANUAL_DIR = Path("local_data/manual")
STANDARDIZED_OUTPUT = MANUAL_DIR / "manual_standardized.csv"

In [10]:
manual_summary = run_manual_upload_extraction(CONFIG, run_id=RUN_ID)
manual_summary

{'stage': 'manual_upload',
 'run_id': 'manual_upload_notebook',
 'input_dir': 'local_data/manual_upload',
 'output_file': 'local_data/manual/manual_extraction_raw.csv',
 'input_files': 4,
 'input_rows': 2529,
 'output_rows': 1835,
 'skipped_empty_text_rows': 887,
 'invalid_json_rows': 0,
 'json_results_extracted': 239,
 'rows_by_file': {'manual01.csv': 477,
  'manual02.csv': 1016,
  'manual03.csv': 103,
  'manual04.csv': 239}}

In [11]:
raw_rows = read_csv(manual_summary["output_file"])
standardized_rows, transform_stats = transform_extraction_rows(raw_rows)

write_csv(STANDARDIZED_OUTPUT, standardized_rows, STANDARDIZED_FIELDS)
standardized_summary = {
    "stage": "manual_standardization",
    "input_file": manual_summary["output_file"],
    "output_file": str(STANDARDIZED_OUTPUT),
    "input_rows": transform_stats.input_rows,
    "output_rows": len(standardized_rows),
    "dropped_empty_text_rows": transform_stats.dropped_empty_text_rows,
    "deduplicated_rows": transform_stats.deduplicated_rows,
    "comments_exploded": transform_stats.comments_exploded,
    "replies_exploded": transform_stats.replies_exploded,
    "invalid_comments_json_rows": transform_stats.invalid_comments_json_rows,
}
write_json(MANUAL_DIR / "manual_standardized_summary.json", standardized_summary)
standardized_summary

{'stage': 'manual_standardization',
 'input_file': 'local_data/manual/manual_extraction_raw.csv',
 'output_file': 'local_data/manual/manual_standardized.csv',
 'input_rows': 1835,
 'output_rows': 1835,
 'dropped_empty_text_rows': 0,
 'deduplicated_rows': 0,
 'comments_exploded': 0,
 'replies_exploded': 0,
 'invalid_comments_json_rows': 0}

In [12]:
standardized_rows[:3]

[{'platform': 'facebook',
  'collection_method': 'manual_upload',
  'id': 'manual01_00002_397d35e2311f',
  'text': 'Yes I support however Ai only helps human to make things faster such as automation of repetition tasks, workflow, process, behavioral temporal analysis and so, however human still govern over AI so that it makes healthcare humanize.',
  'category': 'post',
  'associated_id': 'manual01_00002_397d35e2311f',
  'source_url': 'https://www.facebook.com/photo?fbid=122154865694961632&set=gm.27022735664000042&idorvanity=9612195335480678',
  'provided_language_label': '',
  'provided_classification_label': ''},
 {'platform': 'facebook',
  'collection_method': 'manual_upload',
  'id': 'manual01_00003_4399786712e8',
  'text': 'Your doctor thinks they don\'t need to use AI? 🙄 Cute. No. Arrogant. Thank God they\'re not all prideful like this. I follow a doctor who very confidently says other doctors should soon be considered to be engaging in malpractice if they are not using AI to ass